In [2]:
import os

# 'data' 디렉토리가 없으면 생성합니다.
if not os.path.exists('data'):
    os.makedirs('data')
    print("Created 'data' directory.")
else:
    print("'data' directory already exists.")

Created 'data' directory.


이제 데모를 위한 더미 JSONL 파일을 생성합니다. 실제 시나리오에서는 이 부분을 실제 데이터 로딩 프로세스로 대체해야 합니다.

In [3]:
import json

# train.jsonl을 위한 더미 데이터
train_data = [
    {"dialogue": "Patient: I have a headache. Doctor: How long have you had it?", "soap": "S: Headache. O: Patient states headache. A: Acute headache. P: Advised ibuprofen."},
    {"dialogue": "Patient: My throat hurts. Doctor: Open your mouth please.", "soap": "S: Sore throat. O: Throat appears red. A: Pharyngitis. P: Prescribed antibiotics."}
]

with open('data/train.jsonl', 'w') as f:
    for entry in train_data:
        f.write(json.dumps(entry) + '\n')
print("Created 'data/train.jsonl'.")

# validation.jsonl을 위한 더미 데이터
validation_data = [
    {"dialogue": "Patient: I feel tired. Doctor: Any other symptoms?", "soap": "S: Fatigue. O: No other symptoms. A: General fatigue. P: Advised rest."}
]

with open('data/validation.jsonl', 'w') as f:
    for entry in validation_data:
        f.write(json.dumps(entry) + '\n')
print("Created 'data/validation.jsonl'.")

# test.jsonl을 위한 더미 데이터
test_data = [
    {"dialogue": "Patient: My stomach aches. Doctor: When did it start?", "soap": "S: Stomach ache. O: Abdominal tenderness. A: Gastritis. P: Recommended bland diet."}
]

with open('data/test.jsonl', 'w') as f:
    for entry in test_data:
        f.write(json.dumps(entry) + '\n')
print("Created 'data/test.jsonl'.")

Created 'data/train.jsonl'.
Created 'data/validation.jsonl'.
Created 'data/test.jsonl'.


In [4]:
import json
import csv

class DataTransformer:
    def __init__(self, jsonl_file, output_csv_file):
        self.jsonl_file = jsonl_file
        self.output_csv_file = output_csv_file

    def llama_template(self):
        transformed_data = []
        with open(self.jsonl_file, 'r') as infile:
            for line in infile:
                try:
                    example = json.loads(line)
                    dialogue = example.get("dialogue", "").replace('\n', ' ').strip()
                    soap = example.get("soap", "").replace('\n', ' ').strip()
                    # Llama2 템플릿을 적용합니다.
                    transformed_text = f'<s>[INST] {dialogue} [/INST] {soap} </s>'
                    transformed_data.append({"data": transformed_text})
                except json.JSONDecodeError:
                    print(f"Skipping invalid JSON line: {line}")

        return transformed_data

    def save_to_csv(self, data):
        with open(self.output_csv_file, 'w', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=['data'])
            writer.writeheader()
            writer.writerows(data)

    def process(self):
        transformed_data = self.llama_template()
        self.save_to_csv(transformed_data)
        print(f"Processed data has been saved to {self.output_csv_file}")

files = [
    {'jsonl_file': 'data/train.jsonl', 'output_csv_file': 'data/train_llama_formatted.csv'},
    {'jsonl_file': 'data/validation.jsonl', 'output_csv_file': 'data/validation_llama_formatted.csv'},
    {'jsonl_file': 'data/test.jsonl', 'output_csv_file': 'data/test_llama_formatted.csv'}
]

for file in files:
    transformer = DataTransformer(file['jsonl_file'], file['output_csv_file'])
    transformer.process()

Processed data has been saved to data/train_llama_formatted.csv
Processed data has been saved to data/validation_llama_formatted.csv
Processed data has been saved to data/test_llama_formatted.csv
